<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_13_deep_learning_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13: Introducing Automatic Optimization
### Building a Deep Learning Framework from Scratch

This chapter builds a minimal deep learning framework step by step — tensors, automatic gradient computation (autograd), and automatic optimization — to show what libraries like PyTorch do under the hood.

## 1. Tensors

A Tensor wraps a NumPy array and tracks the gradient computation graph. Every operation on a tensor records enough information to compute gradients during the backward pass.

In [ ]:
import numpy as np

class Tensor:
    def __init__(self, data, autograd=False, creators=None, creation_op=None):
        self.data        = np.array(data, dtype=float)
        self.autograd    = autograd
        self.creators    = creators
        self.creation_op = creation_op
        self.grad        = None

    def backward(self, grad=None):
        if grad is None:
            grad = Tensor(np.ones_like(self.data))
        self.grad = grad
        if self.autograd and self.creators is not None:
            if self.creation_op == "add":
                self.creators[0].backward(grad)
                self.creators[1].backward(grad)
            elif self.creation_op == "mul":
                self.creators[0].backward(Tensor(grad.data * self.creators[1].data))
                self.creators[1].backward(Tensor(grad.data * self.creators[0].data))
            elif self.creation_op == "neg":
                self.creators[0].backward(Tensor(-grad.data))

    def __add__(self, other):
        if self.autograd:
            return Tensor(self.data + other.data, autograd=True,
                          creators=[self, other], creation_op="add")
        return Tensor(self.data + other.data)

    def __mul__(self, other):
        if self.autograd:
            return Tensor(self.data * other.data, autograd=True,
                          creators=[self, other], creation_op="mul")
        return Tensor(self.data * other.data)

    def __neg__(self):
        if self.autograd:
            return Tensor(-self.data, autograd=True,
                          creators=[self], creation_op="neg")
        return Tensor(-self.data)

    def __repr__(self):
        return f"Tensor({self.data})"

a = Tensor([1, 2, 3], autograd=True)
b = Tensor([4, 5, 6], autograd=True)
c = a + b
c.backward(Tensor([1, 1, 1]))

print("a =", a)
print("b =", b)
print("c = a + b =", c)
print("grad of a:", a.grad)
print("grad of b:", b.grad)

## 2. Autograd: Tracking the Computation Graph

Every operation creates a node in the computation graph. During the backward pass, gradients flow back through this graph using the chain rule.

In [ ]:
import numpy as np

class Tensor:
    def __init__(self, data, autograd=False, creators=None, creation_op=None):
        self.data        = np.array(data, dtype=float)
        self.autograd    = autograd
        self.creators    = creators
        self.creation_op = creation_op
        self.grad        = None

    def backward(self, grad=None):
        if grad is None:
            grad = Tensor(np.ones_like(self.data))
        self.grad = grad
        if self.autograd and self.creators:
            if self.creation_op == "add":
                for c in self.creators: c.backward(grad)
            elif self.creation_op == "sub":
                self.creators[0].backward(grad)
                self.creators[1].backward(Tensor(-grad.data))
            elif self.creation_op == "mul":
                self.creators[0].backward(Tensor(grad.data * self.creators[1].data))
                self.creators[1].backward(Tensor(grad.data * self.creators[0].data))
            elif self.creation_op == "mm":
                self.creators[0].backward(Tensor(grad.data.dot(self.creators[1].data.T)))
                self.creators[1].backward(Tensor(self.creators[0].data.T.dot(grad.data)))

    def __add__(self, other):
        return Tensor(self.data + other.data, autograd=self.autograd,
                      creators=[self, other], creation_op="add")

    def __sub__(self, other):
        return Tensor(self.data - other.data, autograd=self.autograd,
                      creators=[self, other], creation_op="sub")

    def mm(self, other):
        return Tensor(self.data.dot(other.data), autograd=self.autograd,
                      creators=[self, other], creation_op="mm")

    def __repr__(self):
        return f"Tensor({np.round(self.data, 4)})"

x  = Tensor([[1.0, 2.0]], autograd=True)
w  = Tensor([[0.5, 0.3], [0.1, 0.8]], autograd=True)
y  = x.mm(w)
y.backward(Tensor([[1.0, 1.0]]))

print("x  =", x)
print("w  =", w)
print("y  = x @ w =", y)
print("grad(w):", w.grad)

## 3. Automatic Optimization with SGD

An optimizer takes parameter tensors and their gradients, then updates them automatically after each backward pass.

In [ ]:
import numpy as np

class SGD:
    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha      = alpha

    def zero(self):
        for p in self.parameters:
            p.grad = None

    def step(self):
        for p in self.parameters:
            if p.grad is not None:
                p.data -= self.alpha * p.grad.data

np.random.seed(0)
weight = type('Tensor', (), {
    'data': np.array([[0.5]]),
    'grad': None
})()

optimizer = SGD([weight], alpha=0.1)

for step in range(5):
    weight.grad = type('Tensor', (), {'data': np.array([[0.2]])})()
    optimizer.step()
    optimizer.zero()
    print(f"Step {step}: weight = {weight.data[0][0]:.4f}")

## 4. Layer Abstraction

Frameworks expose a Layer API that encapsulates weights and their initialization. A Sequential container stacks layers and routes forward/backward passes.

In [ ]:
import numpy as np

class Linear:
    def __init__(self, n_inputs, n_outputs):
        self.weight = np.random.rand(n_inputs, n_outputs) * 0.1
        self.bias   = np.zeros(n_outputs)

    def forward(self, x):
        self.last_input = x
        return x.dot(self.weight) + self.bias

    def backward(self, grad):
        self.grad_w = self.last_input.T.dot(grad)
        self.grad_b = grad.sum(axis=0)
        return grad.dot(self.weight.T)

    def update(self, alpha=0.1):
        self.weight -= alpha * self.grad_w
        self.bias   -= alpha * self.grad_b

def relu(x):       return np.maximum(0, x)
def relu_deriv(x): return (x > 0).astype(float)

np.random.seed(1)
l1 = Linear(3, 4)
l2 = Linear(4, 1)

X = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
y = np.array([[0],[1],[0],[1],[1],[0]], dtype=float)

print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for epoch in range(60):
    a1 = relu(l1.forward(X))
    a2 = l2.forward(a1)

    err   = np.sum((a2 - y) ** 2)
    grad2 = a2 - y
    grad1 = l1.backward(l2.backward(grad2) * relu_deriv(l1.forward(X)))

    l2.update()
    l1.update()

    if epoch % 12 == 0:
        print(f"{epoch:<8} {err:>12.6f}")